# 2. Business analysis with DuckDB

The database has a restaurant table and a deduplicated restaurant–cuisine bridge. Each query below comes directly from the versioned SQL file and generates a table in reports/tables. All rating averages exclude unrated listings.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

import duckdb
from src.pipeline import queries
con = duckdb.connect(str(ROOT/'data/processed/zomato.duckdb'), read_only=True)

In [2]:
for name, sql in queries().items():
    print('\n'+name.upper())
    print(sql.strip())
    result = con.execute(sql).df()
    display(result.head(15))
    print(f'{len(result):,} result rows; full output in reports/tables/{name}.csv')


OVERVIEW
SELECT COUNT(*) AS restaurants, COUNT(rating_for_analysis) AS rated_restaurants,
       AVG(rating_for_analysis) AS mean_rating,
       COUNT(*) FILTER (WHERE rating_for_analysis >= 4) AS highly_rated,
       100.0 * COUNT(*) FILTER (WHERE has_online_delivery = 'Yes') / NULLIF(COUNT(*),0) AS online_delivery_pct,
       100.0 * COUNT(*) FILTER (WHERE has_table_booking = 'Yes') / NULLIF(COUNT(*),0) AS table_booking_pct
FROM restaurants;


,restaurants,rated_restaurants,mean_rating,highly_rated,online_delivery_pct,table_booking_pct
0,9551,7403,3.440024,1380,25.662234,12.124385


1 result rows; full output in reports/tables/overview.csv

HIGH_RATING_BOOKING
SELECT COUNT(*) AS highly_rated,
       COUNT(CASE WHEN has_table_booking = 'Yes' THEN 1 END) AS with_booking,
       100.0 * COUNT(CASE WHEN has_table_booking = 'Yes' THEN 1 END) / NULLIF(COUNT(*),0) AS booking_pct
FROM restaurants WHERE rating_for_analysis >= 4.0;


,highly_rated,with_booking,booking_pct
0,1380,287,20.797101


1 result rows; full output in reports/tables/high_rating_booking.csv

COUNTRY_SUMMARY
SELECT country, COUNT(*) AS restaurants,
       100.0*COUNT(*)/SUM(COUNT(*)) OVER () AS share_pct,
       COUNT(rating_for_analysis) AS rated_restaurants, AVG(rating_for_analysis) AS mean_rating
FROM restaurants GROUP BY country ORDER BY restaurants DESC, country;


...

15 result rows; full output in reports/tables/cost_summary.csv

SERVICE_BY_CITY_PRICE
-- Descriptive strata only; still not a causal estimate. Require 10 rated restaurants per group.
SELECT country, city, price_range, has_table_booking, COUNT(*) AS restaurants,
       COUNT(rating_for_analysis) AS rated_restaurants, AVG(rating_for_analysis) AS mean_rating
FROM restaurants GROUP BY country, city, price_range, has_table_booking
HAVING COUNT(rating_for_analysis) >= 10 ORDER BY country, city, price_range, has_table_booking;


,country,city,price_range,has_table_booking,restaurants,rated_restaurants,mean_rating
0,Brazil,Brasí_lia,4,No,10,10,3.840000
1,Brazil,Rio de Janeiro,4,No,11,11,4.481818
2,Brazil,Sí£o Paulo,4,No,14,14,4.214286
3,India,Ahmedabad,3,No,11,11,4.127273
4,India,Allahabad,3,No,13,13,3.369231
5,India,Amritsar,2,No,11,11,3.636364
6,India,Aurangabad,2,No,15,15,3.340000
7,India,Bhopal,2,No,14,14,3.907143
8,India,Bhubaneshwar,2,No,12,12,3.991667
9,India,Coimbatore,2,No,11,11,4.100000


62 result rows; full output in reports/tables/service_by_city_price.csv

MOST_VOTED_PER_COUNTRY
-- Votes are engagement counts, not sales. Rank within countries with deterministic ties.
WITH ranked AS (
 SELECT country, city, restaurant_name, restaurant_id, votes, rating_for_analysis,
        ROW_NUMBER() OVER (PARTITION BY country ORDER BY votes DESC, restaurant_id) AS vote_rank
 FROM restaurants WHERE is_rated
)
SELECT * FROM ranked WHERE vote_rank <= 5 ORDER BY country, vote_rank;


,country,city,restaurant_name,restaurant_id,votes,rating_for_analysis,vote_rank
0,Australia,Palm Cove,Vivo Bar and Grill,16613649,381,4.4,1
1,Australia,Phillip Island,Mad Cowes Cafe,16604370,351,3.7,2
2,Australia,Tanunda,1918 Bistro & Grill,16608059,339,4.4,3
3,Australia,Beechworth,Bridge Road Brewers,16604911,237,4.6,4
4,Australia,Montville,Poets Cafe,16613059,193,2.4,5
5,Brazil,Sí£o Paulo,Paris 6 Classique,6700402,73,3.4,1
6,Brazil,Sí£o Paulo,Meats,6701257,68,4.3,2
7,Brazil,Sí£o Paulo,Skye - Hotel Unique,6700475,59,4.8,3
8,Brazil,Sí£o Paulo,Veloso,6703176,58,4.6,4
9,Brazil,Sí£o Paulo,Cantina Famiglia Mancini,6706313,49,4.5,5


74 result rows; full output in reports/tables/most_voted_per_country.csv


In [3]:
high = con.execute(queries()['high_rating_booking']).df().iloc[0]
print(f'Among {int(high.highly_rated):,} highly rated restaurants, {int(high.with_booking):,} offer booking: {high.booking_pct:.2f}%.')
con.close()

Among 1,380 highly rated restaurants, 287 offer booking: 20.80%.


## Interpretation

Conditional aggregation answers the booking-share question with the correct denominator. Service comparisons describe an association, not an intervention effect. The city–price strata provide a more comparable view but do not eliminate confounding. Cuisines are overlapping categories, and vote rankings measure engagement rather than revenue.